# 13a Temporal Prediction Feasibility

Leakage-free future-impact ranking workflow. Existing raw inputs are read but not modified.

In [1]:

from pathlib import Path
import hashlib, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

pd.set_option('display.max_columns', 180)
pd.set_option('display.max_rows', 160)
plt.style.use('seaborn-v0_8-whitegrid')
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
REPORT_DIR = PROJECT_ROOT / 'outputs' / 'reports'
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
for d in [TABLE_DIR, FIGURE_DIR, REPORT_DIR, PROC_DIR]: d.mkdir(parents=True, exist_ok=True)
RANDOM_SEED=42
BOOTSTRAPS=2000
PERIODS=[('1967-1985',1967,1985),('1986-2004',1986,2004),('2005-2023',2005,2023)]
TRANSITIONS=[('historical_replication','1967-1985','1986-2004'),('primary_temporal_test','1986-2004','2005-2023')]

def sha256_file(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

def norm_text(x):
    if pd.isna(x): return ''
    s=str(x).strip().lower()
    s=s.replace('&',' and ')
    s=re.sub(r'[^a-z0-9]+',' ',s)
    return re.sub(r'\s+',' ',s).strip()

def split_list_cell(x):
    if pd.isna(x): return []
    s=str(x).strip()
    if not s or s.lower() in {'nan','none'}: return []
    return [p.strip() for p in s.split(',') if p.strip() and p.strip().lower() not in {'nan'}]

def numeric_series(s):
    return pd.to_numeric(s, errors='coerce').astype('float64')

def parse_code(x):
    if pd.isna(x): return ''
    s=str(x).strip()
    if not s or s.lower() in {'nan','none'}: return ''
    if re.fullmatch(r'\d+\.0', s): s=s[:-2]
    return s

def safe_spearman(x,y):
    dat=pd.concat([pd.Series(x),pd.Series(y)],axis=1).replace([np.inf,-np.inf],np.nan).dropna()
    if len(dat)<4 or dat.iloc[:,0].nunique()<2 or dat.iloc[:,1].nunique()<2: return np.nan
    return float(stats.spearmanr(dat.iloc[:,0], dat.iloc[:,1]).statistic)

def safe_kendall(x,y):
    dat=pd.concat([pd.Series(x),pd.Series(y)],axis=1).replace([np.inf,-np.inf],np.nan).dropna()
    if len(dat)<4 or dat.iloc[:,0].nunique()<2 or dat.iloc[:,1].nunique()<2: return np.nan
    return float(stats.kendalltau(dat.iloc[:,0], dat.iloc[:,1]).statistic)

RAW_HASHES_BEFORE={p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}
INV_PATH=RAW_DIR/'India_Flood_Inventory_v3.csv'
BASELINE_PATH=PROC_DIR/'dfsi_verified_baseline.csv'
inv=pd.read_csv(INV_PATH, encoding='utf-8-sig')
baseline=pd.read_csv(BASELINE_PATH)
print('Inventory columns:', inv.columns.tolist())
print('Baseline columns:', baseline.columns.tolist())

COLS={
    'event_id':'UEI','start_date':'Start Date','end_date':'End Date','district':'Districts','state':'State','duration':'Duration(Days)','fatalities':'Human fatality','injuries':'Human injured','affected_population':'Human Displaced','event_source':'Event Source','event_source_id':'Event Souce ID','district_lgd':'District_LGD_Codes','state_codes':'State_Codes','description':'Description of Casualties/injured'
}
for role,col in COLS.items(): assert col in inv.columns, (role,col)
inv['_row_id']=np.arange(len(inv))
inv['_start_dt']=pd.to_datetime(inv[COLS['start_date']], errors='coerce', dayfirst=True)
inv['_end_dt']=pd.to_datetime(inv[COLS['end_date']], errors='coerce', dayfirst=True)
inv['_year']=inv['_start_dt'].dt.year
for col in ['duration','fatalities','injuries','affected_population']:
    inv[f'_{col}_num']=numeric_series(inv[COLS[col]])
inv['_district_list']=inv[COLS['district']].apply(split_list_cell)
inv['_lgd_list']=inv[COLS['district_lgd']].apply(split_list_cell)
inv['_state_list']=inv[COLS['state']].apply(split_list_cell)
inv['_district_count']=inv['_district_list'].apply(len)
inv['_lgd_count']=inv['_lgd_list'].apply(lambda xs: len([parse_code(x) for x in xs if parse_code(x)]))

# Baseline matching universe: one-to-one district records, no full-period DFSI used as predictor.
universe=baseline[baseline['join_status'].eq('one_to_one_comparison_record')].copy().reset_index().rename(columns={'index':'baseline_row_id'})
universe['baseline_district_id']=universe['baseline_row_id'].astype(str)
universe['district_norm']=universe['district_name'].apply(norm_text)
universe['state_norm']=universe['State_Name'].apply(norm_text)
# Build unique LGD map; some baseline rows have comma-separated codes.
lgd_rows=[]
for _,r in universe.iterrows():
    for code in split_list_cell(r.get('available_district_lgd_codes')):
        c=parse_code(code)
        if c: lgd_rows.append({'lgd_code':c,'baseline_district_id':r['baseline_district_id'],'district_name':r['district_name'],'State_Name':r['State_Name']})
lgd_map=pd.DataFrame(lgd_rows)
lgd_unique={}
if not lgd_map.empty:
    for code,g in lgd_map.groupby('lgd_code'):
        if g['baseline_district_id'].nunique()==1:
            lgd_unique[code]=g.iloc[0]['baseline_district_id']
name_key_counts=universe.groupby(['state_norm','district_norm'])['baseline_district_id'].nunique().reset_index(name='n')
name_unique=universe.merge(name_key_counts.query('n==1')[['state_norm','district_norm']],on=['state_norm','district_norm'])
name_map={(r['state_norm'],r['district_norm']):r['baseline_district_id'] for _,r in name_unique.iterrows()}

# Explode event records to listed districts. LGD matching preferred when available.
exploded=[]; unmatched=[]; ambiguous=[]
for _,r in inv.iterrows():
    districts=r['_district_list']; lgds=[parse_code(x) for x in r['_lgd_list']]
    n=max(len(districts),len(lgds),1)
    if len(districts)==0 and len(lgds)==0:
        unmatched.append({'source_row_id':r['_row_id'],'UEI':r[COLS['event_id']],'year':r['_year'],'State':r[COLS['state']],'Districts':r[COLS['district']],'reason':'missing_district_and_lgd'})
        continue
    for i in range(n):
        dname=districts[i] if i < len(districts) else ''
        lgd=lgds[i] if i < len(lgds) else ''
        state_name=r[COLS['state']] if pd.notna(r[COLS['state']]) else ''
        state_norm=norm_text(state_name if len(split_list_cell(state_name))<=1 else '')
        district_norm=norm_text(dname)
        match_id=''; method='unmatched'; status='unmatched'; reason=''
        if lgd and lgd in lgd_unique:
            match_id=lgd_unique[lgd]; method='lgd_code'; status='matched'
        elif lgd and lgd in set(lgd_map['lgd_code']) if not lgd_map.empty else False:
            method='lgd_code'; status='ambiguous'; reason='lgd_code_maps_to_multiple_baseline_districts'
        elif state_norm and district_norm and (state_norm,district_norm) in name_map:
            match_id=name_map[(state_norm,district_norm)]; method='exact_state_district_name'; status='matched'
        elif district_norm:
            candidates=universe[universe['district_norm'].eq(district_norm)]
            if len(candidates)==1:
                match_id=candidates.iloc[0]['baseline_district_id']; method='exact_district_name_only_unique'; status='matched'
            elif len(candidates)>1:
                status='ambiguous'; method='exact_district_name_only'; reason='district_name_matches_multiple_baseline_states'
            else:
                reason='no_exact_lgd_or_name_match'
        else:
            reason='blank_district_name_for_list_position'
        rec={
            'source_row_id':r['_row_id'],'UEI':r[COLS['event_id']],'year':r['_year'],'start_date':r['_start_dt'],'end_date':r['_end_dt'],'event_source':r[COLS['event_source']],
            'event_source_id':r[COLS['event_source_id']],'state_raw':r[COLS['state']],'district_raw':dname,'district_lgd_code':lgd,
            'baseline_district_id':match_id,'match_method':method,'match_status':status,'match_issue':reason,
            'event_district_count':len(districts) if len(districts)>0 else n,'event_lgd_count':len([x for x in lgds if x]),
            'duration_days':r['_duration_num'],'fatalities_reported':r['_fatalities_num'],'injuries_reported':r['_injuries_num'],
            'fatalities_missing':pd.isna(r['_fatalities_num']),'injuries_missing':pd.isna(r['_injuries_num']),'duration_missing':pd.isna(r['_duration_num'])
        }
        exploded.append(rec)
        if status=='unmatched': unmatched.append({**rec,'reason':reason})
        if status=='ambiguous': ambiguous.append({**rec,'reason':reason})
long=pd.DataFrame(exploded)
long=long.merge(universe[['baseline_district_id','State_Name','district_name','population']],on='baseline_district_id',how='left')
# Human impacts are considered district-assignable only for single-district/single-LGD matched rows.
long['district_specific_impact']=long['match_status'].eq('matched') & (long['event_district_count'].le(1)) & (long['event_lgd_count'].le(1))
long['fatalities_for_district_primary']=np.where(long['district_specific_impact'] & long['fatalities_reported'].notna(), long['fatalities_reported'], 0.0)
long['injuries_for_district_primary']=np.where(long['district_specific_impact'] & long['injuries_reported'].notna(), long['injuries_reported'], 0.0)
long.to_csv(TABLE_DIR/'predictive_ranking_event_district_long.csv',index=False)

# Input audit.
dups=int(inv.duplicated(subset=[COLS['event_id'],COLS['start_date'],COLS['end_date'],COLS['district'],COLS['state']],keep=False).sum())
audit=[]
for role,col in COLS.items():
    audit.append({'source_file':str(INV_PATH.relative_to(PROJECT_ROOT)),'variable_role':role,'actual_column':col,'row_count':len(inv),'earliest_year':int(inv['_year'].min()),'latest_year':int(inv['_year'].max()),'missing_count':int(inv[col].isna().sum()),'missing_percent':float(inv[col].isna().mean()*100),'negative_numeric_count':int((numeric_series(inv[col])<0).sum()) if role in ['duration','fatalities','injuries','affected_population'] else np.nan,'duplicated_event_records':dups,'unique_districts_raw':int(pd.Series([d for xs in inv['_district_list'] for d in xs]).nunique()),'unique_states_raw':int(pd.Series([s for xs in inv['_state_list'] for s in xs]).nunique()),'notes':'Column identified from actual India Flood Inventory schema; values preserved.'})
# Add annual fatality/injury and period district representation notes into same audit format.
yearly_imp=inv.groupby('_year',dropna=True).agg(fatalities_sum=('_fatalities_num','sum'),injuries_sum=('_injuries_num','sum'),events=('UEI','count')).reset_index().rename(columns={'_year':'year'})
for _,r in yearly_imp.iterrows():
    audit.append({'source_file':str(INV_PATH.relative_to(PROJECT_ROOT)),'variable_role':'fatalities_and_injuries_by_year','actual_column':'Human fatality; Human injured','row_count':int(r['events']),'earliest_year':int(r['year']),'latest_year':int(r['year']),'missing_count':np.nan,'missing_percent':np.nan,'negative_numeric_count':np.nan,'duplicated_event_records':dups,'unique_districts_raw':np.nan,'unique_states_raw':np.nan,'notes':f"reported fatality sum={r['fatalities_sum']}; reported injury sum={r['injuries_sum']}"})
for label,start,end in PERIODS:
    m=long[long['year'].between(start,end) & long['match_status'].eq('matched')]
    audit.append({'source_file':str(INV_PATH.relative_to(PROJECT_ROOT)),'variable_role':'districts_represented_in_period','actual_column':'Districts; District_LGD_Codes','row_count':len(m),'earliest_year':start,'latest_year':end,'missing_count':np.nan,'missing_percent':np.nan,'negative_numeric_count':np.nan,'duplicated_event_records':dups,'unique_districts_raw':int(m['baseline_district_id'].nunique()),'unique_states_raw':int(m['State_Name'].nunique()),'notes':label})
pd.DataFrame(audit).to_csv(TABLE_DIR/'predictive_ranking_input_audit.csv',index=False)

# Reporting completeness by year and period.
years=pd.DataFrame({'year':range(1967,2024)})
annual=inv.groupby('_year').agg(recorded_events=('UEI','count'),districts_represented=('_district_count',lambda s:int((s>0).sum())),states_represented=('State',lambda s:int(pd.Series([st for v in s for st in split_list_cell(v)]).nunique())),total_fatalities=('_fatalities_num','sum'),total_injuries=('_injuries_num','sum'),duration_nonmissing=('_duration_num',lambda s:s.notna().mean()*100),fatalities_nonmissing=('_fatalities_num',lambda s:s.notna().mean()*100),injuries_nonmissing=('_injuries_num',lambda s:s.notna().mean()*100)).reset_index().rename(columns={'_year':'year','duration_nonmissing':'duration_reported_percent','fatalities_nonmissing':'fatalities_reported_percent','injuries_nonmissing':'injuries_reported_percent'})
annual=years.merge(annual,on='year',how='left').fillna({'recorded_events':0,'districts_represented':0,'states_represented':0,'total_fatalities':0,'total_injuries':0,'duration_reported_percent':0,'fatalities_reported_percent':0,'injuries_reported_percent':0})
annual.to_csv(TABLE_DIR/'predictive_ranking_yearly_completeness.csv',index=False)
period_rows=[]
state_period_rows=[]
for label,start,end in PERIODS:
    p=inv[inv['_year'].between(start,end)]
    lp=long[long['year'].between(start,end)]
    period_rows.append({'period':label,'start_year':start,'end_year':end,'years':end-start+1,'recorded_events':len(p),'events_per_year':len(p)/(end-start+1),'matched_event_district_rows':int((lp['match_status']=='matched').sum()),'matched_districts':int(lp.loc[lp['match_status']=='matched','baseline_district_id'].nunique()),'states_represented':int(lp.loc[lp['match_status']=='matched','State_Name'].nunique()),'total_reported_fatalities_raw_event_level':float(p['_fatalities_num'].sum(skipna=True)),'total_primary_assignable_district_fatalities':float(lp['fatalities_for_district_primary'].sum()),'fatality_reporting_percent':float(p['_fatalities_num'].notna().mean()*100),'injury_reporting_percent':float(p['_injuries_num'].notna().mean()*100),'duration_reporting_percent':float(p['_duration_num'].notna().mean()*100),'multi_district_rows_with_reported_fatalities_excluded_from_primary':int(((lp['event_district_count']>1)&lp['fatalities_reported'].notna()).sum())})
    for state,g in p.groupby('State'):
        state_period_rows.append({'period':label,'State_raw':state,'recorded_events':len(g),'fatality_reporting_percent':float(g['_fatalities_num'].notna().mean()*100),'injury_reporting_percent':float(g['_injuries_num'].notna().mean()*100),'duration_reporting_percent':float(g['_duration_num'].notna().mean()*100),'reported_fatalities':float(g['_fatalities_num'].sum(skipna=True))})
period_df=pd.DataFrame(period_rows); period_df.to_csv(TABLE_DIR/'predictive_ranking_period_completeness.csv',index=False)
pd.DataFrame(state_period_rows).to_csv(TABLE_DIR/'predictive_ranking_state_period_completeness.csv',index=False)

match_audit=pd.DataFrame([{'audit_item':'event_district_rows','value':len(long),'notes':'Rows after exploding listed districts/LGD codes.'},{'audit_item':'matched_rows','value':int((long['match_status']=='matched').sum()),'notes':'LGD preferred; exact names only when unique.'},{'audit_item':'unmatched_rows','value':int((long['match_status']=='unmatched').sum()),'notes':'Preserved; not used in panel.'},{'audit_item':'ambiguous_rows','value':int((long['match_status']=='ambiguous').sum()),'notes':'Preserved; not used in panel.'},{'audit_item':'single_district_primary_impact_rows','value':int(long['district_specific_impact'].sum()),'notes':'Only these rows assign fatalities/injuries to districts in primary target.'}])
match_audit.to_csv(TABLE_DIR/'predictive_ranking_district_match_audit.csv',index=False)
pd.DataFrame(unmatched).to_csv(TABLE_DIR/'predictive_ranking_unmatched_events.csv',index=False)
pd.DataFrame(ambiguous).to_csv(TABLE_DIR/'predictive_ranking_ambiguous_districts.csv',index=False)

# Feasibility transition summaries.
feas=[]
for name,pred_period,out_period in TRANSITIONS:
    pred_start,pred_end=[(s,e) for label,s,e in PERIODS if label==pred_period][0]
    out_start,out_end=[(s,e) for label,s,e in PERIODS if label==out_period][0]
    pred=long[long['year'].between(pred_start,pred_end)&long['match_status'].eq('matched')]
    out=long[long['year'].between(out_start,out_end)&long['match_status'].eq('matched')]
    common=set(pred['baseline_district_id'])|set(out['baseline_district_id'])
    out_fat=out.groupby('baseline_district_id')['fatalities_for_district_primary'].sum()
    feas.append({'transition':name,'predictor_period':pred_period,'outcome_period':out_period,'predictor_matched_districts':pred['baseline_district_id'].nunique(),'outcome_matched_districts':out['baseline_district_id'].nunique(),'districts_with_either_period_records':len(common),'outcome_districts_with_positive_primary_fatalities':int((out_fat>0).sum()),'total_primary_assignable_future_fatalities':float(out_fat.sum()),'outcome_fatality_variance':float(out_fat.var()) if len(out_fat)>1 else np.nan})
feas_df=pd.DataFrame(feas); feas_df.to_csv(TABLE_DIR/'predictive_ranking_feasibility_decision.csv',index=False)

# Figure.
fig,axes=plt.subplots(3,1,figsize=(10,8),sharex=True,constrained_layout=True)
axes[0].plot(annual['year'],annual['recorded_events'],color='#1f4e79'); axes[0].set_ylabel('Recorded events')
axes[1].plot(annual['year'],annual['fatalities_reported_percent'],label='Fatalities',color='#b55'); axes[1].plot(annual['year'],annual['injuries_reported_percent'],label='Injuries',color='#579'); axes[1].plot(annual['year'],annual['duration_reported_percent'],label='Duration',color='#595'); axes[1].set_ylabel('Reported (%)'); axes[1].legend(fontsize=8)
axes[2].plot(annual['year'],annual['total_fatalities'],label='Fatalities',color='#b55'); axes[2].plot(annual['year'],annual['total_injuries'],label='Injuries',color='#579'); axes[2].set_ylabel('Reported total'); axes[2].set_xlabel('Year'); axes[2].legend(fontsize=8)
for ax in axes:
    for x in [1985.5,2004.5]: ax.axvline(x,color='black',linewidth=.8,alpha=.5)
fig.suptitle('Flood-inventory reporting volume, impact reporting and recorded impacts over time')
fig.savefig(FIGURE_DIR/'predictive_ranking_reporting_over_time.png',dpi=220,bbox_inches='tight'); plt.close(fig)
fig.savefig(FIGURE_DIR/'predictive_ranking_reporting_completeness.png',dpi=220,bbox_inches='tight') if False else None
# copy by re-rendering simpler duplicate for required later figure name
fig,ax=plt.subplots(figsize=(10,4.5),constrained_layout=True)
ax.plot(annual['year'],annual['recorded_events'],label='Recorded events',color='#1f4e79'); ax2=ax.twinx(); ax2.plot(annual['year'],annual['fatalities_reported_percent'],label='Fatality reporting %',color='#b55'); ax.set_ylabel('Recorded events'); ax2.set_ylabel('Fatality reporting (%)'); ax.set_xlabel('Year'); ax.set_title('Reporting volume and fatality completeness over time'); fig.savefig(FIGURE_DIR/'predictive_ranking_reporting_completeness.png',dpi=220,bbox_inches='tight'); plt.close(fig)

# Feasibility gate thresholds are project decision rules, not universal standards.
usable_dates=inv['_year'].between(1967,2023).mean()>0.98
credible_matching=(long['match_status'].eq('matched').mean()>0.75)
meaningful_variation=(feas_df['outcome_districts_with_positive_primary_fatalities'].min()>=40 and feas_df['total_primary_assignable_future_fatalities'].min()>200)
consecutive_overlap=(feas_df['districts_with_either_period_records'].min()>=300)
not_meaningless_reporting=period_df['fatality_reporting_percent'].min()>45
feasibility_status='PROCEED' if all([usable_dates,credible_matching,meaningful_variation,consecutive_overlap,not_meaningless_reporting]) else 'STOP'
feasibility_notes=pd.DataFrame([{'criterion':'usable_event_dates','passed':usable_dates,'notes':'Valid year within 1967-2023 for nearly all records.'},{'criterion':'credible_district_matching','passed':credible_matching,'notes':'LGD/exact unique matching covers most exploded district rows.'},{'criterion':'fatalities_have_meaningful_variation','passed':meaningful_variation,'notes':'Uses only district-specific/single-district assignable fatalities.'},{'criterion':'enough_consecutive_period_districts','passed':consecutive_overlap,'notes':'Districts represented across predictor/outcome periods.'},{'criterion':'reporting_not_meaningless','passed':not_meaningless_reporting,'notes':'Fatality reporting remains incomplete and drifting; proceed only with cautions if passed.'},{'criterion':'overall_feasibility_status','passed':feasibility_status=='PROCEED','notes':feasibility_status}])
feasibility_notes.to_csv(TABLE_DIR/'predictive_ranking_feasibility_status.csv',index=False)
print('Feasibility status:', feasibility_status)
display(period_df)
display(feas_df)
display(feasibility_notes)
RAW_HASHES_AFTER={p.name: sha256_file(p) for p in sorted(RAW_DIR.glob('*.csv'))}
assert RAW_HASHES_BEFORE==RAW_HASHES_AFTER, 'data/raw changed during feasibility audit.'


Inventory columns: ['Unnamed: 0', 'UEI', 'Start Date', 'End Date', 'Duration(Days)', 'Main Cause', 'Location', 'Districts', 'State', 'Latitude', 'Longitude', 'Severity', 'Area Affected', 'Human fatality', 'Human injured', 'Human Displaced', 'Animal Fatality', 'Description of Casualties/injured', 'Extent of damage ', 'Event Source', 'Event Souce ID', 'District_LGD_Codes', 'State_Codes']
Baseline columns: ['State_Name', 'district_name', 'available_state_identifier', 'available_state_review_key', 'available_district_identifier', 'available_district_review_key', 'available_district_lgd_codes', 'available_district_lgd_code_count', 'inventory_district_names', 'dfsi_source_row_id', 'impact_source_row_id', 'area_source_row_id', 'flood_event_count', 'mean_flood_duration', 'fatalities', 'injured', 'corrected_percent_flooded_area', 'population', 'published_dfsi', 'reconstructed_dfsi', 'reconstructed_dfsi_product_space', 'log_product_agreement_abs_diff', 'published_rank', 'reconstructed_rank', 'nu

Feasibility status: PROCEED


,period,start_year,end_year,years,recorded_events,events_per_year,matched_event_district_rows,matched_districts,states_represented,total_reported_fatalities_raw_event_level,total_primary_assignable_district_fatalities,fatality_reporting_percent,injury_reporting_percent,duration_reporting_percent,multi_district_rows_with_reported_fatalities_excluded_from_primary
0,1967-1985,1967,1985,19,762,40.105263,4432,550,31,16933.0,3201.0,49.868766,14.173228,100.0,2300
1,1986-2004,1986,2004,19,1968,103.578947,6710,549,32,29011.0,3202.0,61.839431,17.784553,100.0,5714
2,2005-2023,2005,2023,19,4126,217.157895,9373,568,33,24745.0,3654.0,52.641784,14.493456,100.0,5440


,transition,predictor_period,outcome_period,predictor_matched_districts,outcome_matched_districts,districts_with_either_period_records,outcome_districts_with_positive_primary_fatalities,total_primary_assignable_future_fatalities,outcome_fatality_variance
0,historical_replication,1967-1985,1986-2004,550,549,579,162,3202.0,508.665297
1,primary_temporal_test,1986-2004,2005-2023,549,568,604,332,3654.0,389.544017


,criterion,passed,notes
0,usable_event_dates,True,Valid year within 1967-2023 for nearly all rec...
1,credible_district_matching,True,LGD/exact unique matching covers most exploded...
2,fatalities_have_meaningful_variation,True,Uses only district-specific/single-district as...
3,enough_consecutive_period_districts,True,Districts represented across predictor/outcome...
4,reporting_not_meaningless,True,Fatality reporting remains incomplete and drif...
5,overall_feasibility_status,True,PROCEED
